In [4]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [5]:
!pip install -q openai langchain-openai langchain-community faiss-cpu python-dotenv gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [8]:
# 환경 설정
import os
from dotenv import load_dotenv
import pandas as pd
import json
import time
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")

In [19]:
plain_prompt = """다음 제품 리뷰를 분석해줘, 전체 감정(긍정/부정/혼합), 1-5점 점수, 장점 목록
단점 목록, 핵심 키워드 3개를 알려줘, 리뷰: "이 노트북 정말 가벼워서 좋아요! 하지만 키보드 타건감이 아쉽네요.
유효한 json만 출력하세요"""

print(llm.invoke([HumanMessage(content=plain_prompt)]).content)

```json
{
  "overall_sentiment": "혼합",
  "score": 4,
  "pros": [
    "가벼운 무게"
  ],
  "cons": [
    "타건감 아쉬움"
  ],
  "keywords": [
    "가벼움",
    "키보드",
    "타건감"
  ]
}
```


In [20]:
markdown_prompt = """# 제품 리뷰 감정 분석
## 입력 리뷰
>  "이 노트북 정말 가벼워서 좋아요! 하지만 키보드 타건감이 아쉽네요."

## 분석 항목
- **overall_sentiment**: 긍정/부정/혼합 중 하나
- **score** : 1-5 점수
- **pros** : 장점 리스트
- **cons** : 단점 리스트
- **keywords** : 핵심 키워드 3개

## 출력 형식
유효한 json만 출력하세요. 다른 텍스트를 포함하지 마세요.
"""

print(llm.invoke([HumanMessage(content=markdown_prompt)]).content)

# 얘는 ```json... ```으로 안줌

```json
{
  "overall_sentiment": "혼합",
  "score": 4,
  "pros": [
    "가벼움"
  ],
  "cons": [
    "키보드 타건감 부족"
  ],
  "keywords": [
    "노트북",
    "가벼움",
    "키보드"
  ]
}
```


In [16]:
json_prompt = """다음 제품 리뷰를 분석해서 json 형식으로 출력하세요

리뷰 : "  "이 노트북 정말 가벼워서 좋아요! 하지만 키보드 타건감이 아쉽네요."

출력 형식
{

  "overall_sentiment" 긍정/부정/혼합 중 하나,
  "score" : 1-5 점수,
  "pros" : [장점] ,
  "cons" : [단점] ,
  "keywords" : [키워드1, 키워드2, 키워드3],

}
"""
print(llm.invoke([HumanMessage(content=json_prompt)]).content)

# 출력이 ```json ... ``` 형태로 나옴

```json
{
  "overall_sentiment": "혼합",
  "score": 3,
  "pros": ["가벼움"],
  "cons": ["키보드 타건감 아쉬움"],
  "keywords": ["노트북", "가벼움", "키보드"]
}
```


In [15]:
# json 등, 형식을 제한하면 성능(답변품질)이 제한될 수 있으나 파싱해서 사용하기 편함

오히려 format을 제한할 경우 성능이 올라갈 수 있다는 연구 케이스
- 참고: https://huggingface.co/blog/evaluation-structured-outputs

In [21]:
# 회의록 -> 구조화된 데이터로 변환

json_structure = {
    'date' : 'YYYY-MM-DD',
    'attendees' : ['이름1', '이름2'],
    'agenda' : ['안건1', '안건2'],
    'decisions' : ['결정1'],
    'action_items' : [{'assignee':'담당자', 'task': '작업', 'deadline': '기한'}]

}

In [84]:
text = """
3월 15일 마케팅팀 주간 회의.
참석: 김팀장, 이대리, 박사원. 신규 SNS 캠페인 예산 5000만원 확정.
이대리가 3월 22일까지 시안 준비. 박사원은 경쟁사 분석 보고서 3월 20일까지.
"""


def generate_meeting_minute(raw_text):
  messages = [
      SystemMessage(content = '회의록을 정리해주세요.'),
      HumanMessage(content = f"""다음 회의 내용을 지정된 json 구조로 알려주세요, 회의내용: {raw_text}

                          json 구조:
                          'date' : 'YYYY-MM-DD',
                          'attendees' : ['이름1', '이름2'],
                          'agenda' : ['안건1', '안건2'],
                          'decisions' : ['결정1'],
                          'action_items' : {{'assignee':'담당자', 'task': '작업', 'deadline': '기한'}}

      """)

  ]
# 파이썬의 f-string에서 중괄호 {}를 일반 문자열로 인식하게 하려면
# 백슬래시(\)가 아니라 중괄호를 두 번 겹쳐서 {{ 와 }} 로 작성해야 함.
# 코드의 action_items 부분에 있는 중괄호를 두 번씩 쓰면 오류 해결.
  print(llm.invoke(messages).content)

In [85]:
generate_meeting_minute(text)

```json
{
    "date": "2023-03-15",
    "attendees": ["김팀장", "이대리", "박사원"],
    "agenda": ["신규 SNS 캠페인 예산 확정", "경쟁사 분석 보고서"],
    "decisions": ["신규 SNS 캠페인 예산 5000만원 확정"],
    "action_items": [
        {
            "assignee": "이대리",
            "task": "시안 준비",
            "deadline": "2023-03-22"
        },
        {
            "assignee": "박사원",
            "task": "경쟁사 분석 보고서 작성",
            "deadline": "2023-03-20"
        }
    ]
}
```


In [86]:
# prompt: 제약조건
# 길이 : 3문장으로 요약, 100자 이내
# 내용: 검색된 정보로만 얘기, 주어진 정보 안에서만 얘기하세요, 추측하지 마세요.
# 형식: json 형태로만 출력하세요, 불릿포인트만 사용해서 출력하세요

# 부정제약보단 긍정제약이 유효
  # 부정제약: 거짓말로 지어내지마세요
  # 긍정제약: 사실만 그대로 이야기하세요

💡 메서드 체이닝(Method Chaining) 핵심 요약
- 여러 개의 메서드를 점(.)으로 기차처럼 길게 이어서 한 번에 실행하는 방식입니다. (예: 객체.동작1().동작2())
- 이렇게 계속 이어서 호출하려면, 각 메서드가 작업 후 반드시 자기 자신(return self)을 반환해야 합니다.
- 매번 변수를 새로 선언할 필요가 없어져 코드가 훨씬 간결하고 자연스럽게 읽힙니다.

In [110]:
class ConstrainedPrompt:
  def __init__(self, base_instruction):
    self.instruction = base_instruction
    self.constraints = []

  # 길이, 컨텐트, 포맷 제약
  def add_length(self, description):
    self.constraints.append(f"[길이] {description}")
    return self # return self -> 메서드 체이닝

  def add_content(self, description):
    self.constraints.append(f"[내용] {description}")
    return self

  def add_format(self, description):
    self.constraints.append(f"[형식] {description}")
    return self

  def add_style(self, description):
    self.constraints.append(f"[스타일] {description}")
    return self

  def build(self):
    parts = [self.instruction, "\n제약조건:"]
    for c in self.constraints:
      parts.append(f" - {c}")
    return '\n'.join(parts)

  def execute(self, **kwargs):
    #print('kwargs', kwargs) # kwargs, 뭘 받을지 모르는 상태
    prompt = self.build()
    return llm.invoke([HumanMessage(content=prompt)]).content

In [105]:
result = (
    ConstrainedPrompt('클라우드 컴퓨팅의 장점을 설명해주세요')
    .add_length("5개의 불릿포인트")
    .add_content('비용, 확장성, 보안 관점을 반드시 포함')
    .add_format('각 포인트는 한줄로, 이모지로 시작')
    .add_style('IT 비전공 경영진을 대상, 전문 용어에 괄호로 설명 추가')
    .execute(temperature=0.3) # def execute에, temperature 인자를 따로 명시 안했는데, 입력이 가능함 -> **kwargs 때문임.
)

kwargs {'temperature': 0.3}


In [106]:
print(result)

- 💰 **비용 절감**: 클라우드 서비스는 초기 투자 비용을 줄이고 필요에 따라 사용량에 따라 결제할 수 있어 경제적입니다.
- 📈 **확장성**: 비즈니스 성장에 따라 서버 용량을 쉽게 조정할 수 있어 빠른 대응이 가능합니다.
- 🔒 **보안 강화**: 클라우드 제공업체들은 전문적인 보안 시스템을 갖추고 있어 데이터 보호가 용이합니다.
- 🌐 **접근성**: 인터넷만 있으면 언제 어디서나 데이터와 애플리케이션에 접근할 수 있어 유연성이 높습니다.
- 🤝 **협업 촉진**: 여러 팀원이 동시에 작업할 수 있어 효율적인 협업과 의사소통이 가능합니다.


#### 실습: ConstraiendPrompt를 이용해 광고카피 및 상세설명 항목으로 구성된 답변 만들기

In [108]:
product = "에어프로 맥스 무선 헤드폰"
features = ["40시간 배터리", "멀티포인트 연결", "30dB 노이즈캔슬링", "300g 경량 설계"]

In [113]:
answer = (
    ConstrainedPrompt(f"{product}의 장점을 설명해주세요. 특징: {'. '.join(features)}")
    .add_length("5개의 불릿 포인트")
    .add_content(f"핵심 특징 항목에 대한 설명을 포함")
    .add_format('각 포인트는 한줄로, 이모지로 시작')
    .add_style('대중 및 주요 소비자를 대상으로, 장점을 극대화 할 수 있도록')
    .execute()
)

In [114]:
print(answer)

- 🎧 **40시간 배터리**: 긴 배터리 시간을 제공하여 하루 종일 음악이나 통화를 자유롭게 즐길 수 있습니다.  
- 📶 **멀티포인트 연결**: 두 기기와 동시에 연결 가능해, 스마트폰과 노트북을 번거롭지 않게 전환할 수 있어요.  
- 🔇 **30dB 노이즈캔슬링**: 강력한 노이즈 캔슬링 기능으로 외부 소음을 차단해 몰입감을 극대화합니다.  
- 🪶 **300g 경량 설계**: 가벼운 디자인 덕분에 장시간 착용해도 편안함을 유지할 수 있습니다.  
- 🎶 **탁월한 음질**: 생생한 사운드와 깊은 베이스로 음악을 더욱 즐겁게 감상할 수 있습니다.  


In [115]:
# 컨텍스트 제공
# RAG: llm의 실시간성
# 근데 context는 너무 길어지면 성능이 떨어지기 시작함


### RAG: llm의 실시간성
- 근데 context는 너무 길어지면 성능이 떨어지기 시작함
- 관련 논문: https://arxiv.org/pdf/2307.03172
  - 위치에 대해서, 맨앞이나 끝부분에 위치할때 accuracy 높음, 중앙일때 최저

In [122]:
company_policy = """
[모두컴퍼니 재택근무 정책 v2.3]
- 주 3일 재택, 2일 출근 (화/목 필수 출근)
- 재택근무 시 오전 9시까지 Slack 상태 '업무중' 설정 필수
- 해외 원격근무는 최대 연속 2주까지 가능 (사전 승인 필요)
- 야간근무(22시 이후) 시 익일 오후 출근 가능
- 재택근무 장비 지원금: 연 100만원 (영수증 제출)
"""

In [123]:
question = '해외에서 한 달 동안 원격 근무할 수 있나요?'

In [131]:
# 컨텍스트를 제공해 넣기

with_context = f"""아래 회사 정책 문서를 참고하여 질문에 답하세요.
문서에 없는 내용은 "해당 정책 문서에 명시되어 있지 않습니다." 라고 답하세요.

정책 문서:

\"\"\"{company_policy}\"\"\"


질문 : {question}"""
print(llm.invoke([HumanMessage(content=with_context)]).content)

해당 정책 문서에 명시되어 있지 않습니다.


In [129]:
# 컨텍스트 제공 및 답변형식, 근거 제약 등 부여

def answer_with_context(context, question):
  clause = """
  중요: 반드시 제공된 문서만을 근거로 답변하세요.
  문서에 없는 내용은 "제공된 문서에 해당 정보가 없습니다." 라고 답하세요.
  추측하거나 외부 지식을 사용하지 마세요."""

  prompt = f"""아래 참고 문서를 기반으로 질문에 답하세요.
  {clause}

  참고 문서:
  \"\"\"{company_policy}\"\"\"

  질문: {question}

  답변 형식:
  - 답변 : [핵심 답변]
  - 근거: [문서에서 관련 부분 인용]"""


  return print(llm.invoke([HumanMessage(content=prompt)]).content)


In [130]:
answer_with_context(company_policy, question) # 컨텍스트 및 clause 유무에 따라서 답변이 차이남

- 답변 : 아니요, 한 달 동안 원격 근무할 수 없습니다.
- 근거: "해외 원격근무는 최대 연속 2주까지 가능 (사전 승인 필요)"


In [132]:
answer_with_context(company_policy, question= '재택근무는 몇일까지 되나요')

- 답변 : 재택근무는 주 3일 가능하다.
- 근거: "[모두컴퍼니 재택근무 정책 v2.3] - 주 3일 재택, 2일 출근 (화/목 필수 출근)"


In [133]:
answer_with_context(company_policy, question= '식대 지원금은 얼마인가요?')

- 답변 : 제공된 문서에 해당 정보가 없습니다.
- 근거: 제공된 문서에는 식대 지원금에 대한 내용이 포함되어 있지 않습니다.


### 프롬프트 엔지니어링에서 사람의 직관이 차지하는 부분이 너무 많고, 랜덤성이 높다, 정확한 수식이나 이론 등이 없다..
- 관련 논문: https://arxiv.org/pdf/2211.01910
- 'APE(automatic prompt engineer)' 라는 것을 제시, 노가다로 프롬프트 엔지니어링을 하던 것을 자동 및 정량화 등등 자동화.
- 프롬프트도 LLM을 이용하면 더 잘할 수 있다 (프롬프트 작성 자동화).

In [134]:
# zero-shot, few-shot
  # zero-shot: '식대 지원금은 얼마인가요?' -> 감정 분석해주세요/
  # few-shot : ''식대 지원금은 얼마인가요?' -> 추가정보 ('긍정': 추가내용) -> 감성 분석해주세요.



In [136]:
categories = ['기술', '경제', '스포츠', '문화', '정치']

news_articles = [
    "삼성전자가 차세대 AI 반도체 개발에 3조원을 투자한다고 발표했다.",
    "한국은행이 기준금리를 0.25%p 인하하며 경기 부양에 나섰다.",
    "손흥민이 프리미어리그 시즌 최다 도움을 기록하며 팀 승리를 이끌었다.",
    "국립현대미술관에서 한국 현대미술 50년 특별전이 개막했다."
]

In [137]:
classify_prompt = f"""당신은 뉴스 분류 전문가입니다.
주어진 뉴스 기사를 다음 카테고리 중 하나로 분류 하세요 {'. '.join(categories)}
반드시 카테고리 이름만 출력하세요"""

for article in news_articles:
  result = llm.invoke([
      SystemMessage(content = classify_prompt),
      HumanMessage(content = article)
  ]).content
  print(f"기사 :{article[:30]}...")
  print(f"분류 :{result}\n")

기사 :삼성전자가 차세대 AI 반도체 개발에 3조원을 투자한다...
분류 :기술

기사 :한국은행이 기준금리를 0.25%p 인하하며 경기 부양에...
분류 :경제

기사 :손흥민이 프리미어리그 시즌 최다 도움을 기록하며 팀 승...
분류 :스포츠

기사 :국립현대미술관에서 한국 현대미술 50년 특별전이 개막했...
분류 :문화



#### Few-shot

- AIMessage는 "AI가 이렇게 대답해야 해!" 하고 미리 보여주는 '모범 답안(예시)' 역할을 합니다.
- 코드 안에 있는 세 개의 AIMessage를 보면 모두 공통적인 특징이 있습니다.
  ```
  {"sentiment": "긍정", "score": 0.95, "keywords": ["최고", "강력 추천"]}
  {"sentiment": "부정", "score": 0.15, "keywords": ["느리고", "형편 없네요"]}
  {"sentiment": "혼합", "score": 0.5, "keywords":["괜찮지만", "아쉬워요"]}
  ```
- 이 AIMessage들의 목적은 다음과 같습니다:
  - 출력 형식 지정: AI에게 "답변을 할 때는 반드시 줄글이 아니라 JSON 형식({ } 모양)으로 대답해"라고 알려줍니다.
  - 항목 지정: JSON 안에 sentiment(감정), score(점수), keywords(핵심 키워드)라는 3가지 키를 꼭 넣어서 답변하도록 학습시킵니다.
  - 퓨샷 프롬프팅(Few-shot Prompting): 이렇게 사람의 질문(HumanMessage)과 AI의 이상적인 답변(AIMessage)을 세트로 묶어서 몇 가지 예시를 던져주면, AI가 눈치껏 패턴을 파악하게 됩니다.

In [142]:
few_shot_messages = [
    SystemMessage(content = '주어진 리뷰의 감정을 분석하세요'),
    # 예시
    HumanMessage(content = "리뷰': '이 제품 정말 최고입니다! 강력 추천해요'"),
    AIMessage(content = '{"sentiment": "긍정", "score": 0.95, "keywords": ["최고", "강력 추천"]}'),
        # 예시
    HumanMessage(content = "리뷰': '배송도 느리고 제품 품질도 안좋아요'"),
    AIMessage(content = '{"sentiment": "부정", "score": 0.15, "keywords": ["느리고", "형편 없네요"]}'),
            # 예시
    HumanMessage(content = "리뷰': '가격 대비 괜찮지만 무난해요'"),
    AIMessage(content = '{"sentiment": "혼합", "score": 0.5, "keywords": ["괜찮지만", "아쉬워요"]}'),

    # 실제 쿼리
    HumanMessage(content= "리뷰: '디자인은 예쁜데, 배터리가 빨리 닳아요, 전체적으로 보통입니다.")
]

print(llm.invoke(few_shot_messages).content)

{"sentiment": "혼합", "score": 0.4, "keywords": ["예쁜", "배터리 빨리 닳아요", "전체적으로 보통"]}


#### 실습: 비공식체를 공식문체로 바꿔주는 프롬프트 구성

In [143]:
examples = [
    {"informal": "내일 미팅 좀 미룰 수 있을까? 갑자기 일이 생겼어.",
     "formal": "안녕하세요. 내일 예정된 미팅 일정 변경을 요청드립니다. 긴급한 업무가 발생하여 조율이 필요합니다. 가능한 대체 일정을 알려주시면 감사하겠습니다."},
    {"informal": "그 보고서 다 했어? 빨리 보내줘.",
     "formal": "안녕하세요. 요청드렸던 보고서 진행 상황을 확인드립니다. 완료되셨다면 전달 부탁드리며, 추가 시간이 필요하시면 말씀해 주세요."},
    {"informal": "이번 프로젝트 결과 별로인데 어떻게 할까?",
     "formal": "안녕하세요. 이번 프로젝트 결과에 대해 논의가 필요합니다. 개선 방안을 함께 검토하기 위해 미팅을 잡는 것이 어떨까요?"}
]

In [150]:
test_messages = [
    "다음 주 워크숍 참석 못 할 것 같아. 다른 사람 보내도 돼?",
    "예산 좀 더 받을 수 있을까? 지금 부족해."
]

In [155]:
examples = [
    {"informal": "내일 미팅 좀 미룰 수 있을까? 갑자기 일이 생겼어.",
     "formal": "안녕하세요. 내일 예정된 미팅 일정 변경을 요청드립니다. 긴급한 업무가 발생하여 조율이 필요합니다. 가능한 대체 일정을 알려주시면 감사하겠습니다."},
    {"informal": "그 보고서 다 했어? 빨리 보내줘.",
     "formal": "안녕하세요. 요청드렸던 보고서 진행 상황을 확인드립니다. 완료되셨다면 전달 부탁드리며, 추가 시간이 필요하시면 말씀해 주세요."},
    {"informal": "이번 프로젝트 결과 별로인데 어떻게 할까?",
     "formal": "안녕하세요. 이번 프로젝트 결과에 대해 논의가 필요합니다. 개선 방안을 함께 검토하기 위해 미팅을 잡는 것이 어떨까요?"}
]



few_shot_messages = [
    SystemMessage(content = '주어진 텍스트를 공손한 문체(formal expression)로 변환해주세요'),
    # 예시
    HumanMessage(content = f"'informal': {examples[0].get('informal')}"),
    AIMessage(content = f"'formal': {examples[0].get('formal')}"),
    # 예시
    HumanMessage(content = f"'informal': {examples[1].get('informal')}"),
    AIMessage(content = f"'formal': {examples[1].get('formal')}"),
    # 예시
    HumanMessage(content = f"'informal': {examples[2].get('informal')}"),
    AIMessage(content = f"'formal': {examples[2].get('formal')}"),

    # 실제 쿼리
    HumanMessage(content= test_messages[0])
]

print(llm.invoke(few_shot_messages).content)

안녕하세요. 다음 주 예정된 워크숍에 참석하지 못할 것 같습니다. 대신 다른 분이 참석하셔도 되는지 여쭙고 싶습니다. 확인 부탁드립니다. 감사합니다.


'내일 미팅 좀 미룰 수 있을까? 갑자기 일이 생겼어.'